# Experiment Tracking: Organizing ML Experiments at Scale

## Learning Objectives
- Implement experiment tracking for hyperparameter search
- Organize and compare 1000s of experiments
- Find and reproduce best models from past experiments
- Know tools and when to use each

## Interview Questions This Covers
- "You ran 100 experiments. How do you find and compare them?"
- "Achieved 95% accuracy 2 months ago. How do you find that run?"
- "How do you organize experiments across your team?"

## Basic Implementation: Simple Experiment Logger

In [ ]:
import json
from datetime import datetime
from typing import Dict, List, Any
from pathlib import Path

class ExperimentTracker:
    """Simple experiment tracking (JSON-based)"""
    
    def __init__(self, log_dir: str = "experiments"):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(exist_ok=True)
        self.experiments = []
    
    def start_run(self, experiment_name: str, tags: Dict[str, str] = None) -> str:
        """Start a new experiment run"""
        run_id = f"{experiment_name}_{int(datetime.now().timestamp())}"
        self.current_run = {
            'run_id': run_id,
            'experiment_name': experiment_name,
            'start_time': datetime.now().isoformat(),
            'parameters': {},
            'metrics': {},
            'tags': tags or {},
        }
        return run_id
    
    def log_param(self, key: str, value: Any):
        """Log a parameter"""
        if hasattr(self, 'current_run'):
            self.current_run['parameters'][key] = value
    
    def log_metric(self, key: str, value: float, step: int = 0):
        """Log a metric"""
        if hasattr(self, 'current_run'):
            if key not in self.current_run['metrics']:
                self.current_run['metrics'][key] = []
            self.current_run['metrics'][key].append({'step': step, 'value': value})
    
    def end_run(self):
        """End current run and save"""
        if hasattr(self, 'current_run'):
            self.current_run['end_time'] = datetime.now().isoformat()
            
            # Save to file
            run_file = self.log_dir / f"{self.current_run['run_id']}.json"
            with open(run_file, 'w') as f:
                json.dump(self.current_run, f, indent=2)
            
            self.experiments.append(self.current_run)
            delattr(self, 'current_run')
    
    def get_best_run(self, metric: str, mode: str = 'max') -> Dict:
        """Find best run by metric"""
        best = None
        best_value = -float('inf') if mode == 'max' else float('inf')
        
        for exp in self.experiments:
            if metric in exp['metrics']:
                value = exp['metrics'][metric][-1]['value']  # Last value
                if (mode == 'max' and value > best_value) or (mode == 'min' and value < best_value):
                    best = exp
                    best_value = value
        
        return best
    
    def list_experiments(self, tags_filter: Dict[str, str] = None) -> List[Dict]:
        """List experiments with optional filtering"""
        results = self.experiments
        
        if tags_filter:
            for key, value in tags_filter.items():
                results = [r for r in results if r['tags'].get(key) == value]
        
        return results

# Usage
tracker = ExperimentTracker()

# Simulate hyperparameter search: learning_rate × batch_size
for lr in [0.001, 0.01, 0.1]:
    for bs in [32, 64, 128]:
        tracker.start_run('hyperparams_v1', tags={'model': 'xgboost'})
        tracker.log_param('learning_rate', lr)
        tracker.log_param('batch_size', bs)
        
        # Simulate training
        simulated_accuracy = 0.85 + (0.01 * lr) + (0.0001 * bs)
        tracker.log_metric('accuracy', simulated_accuracy)
        tracker.log_metric('loss', 1.0 - simulated_accuracy)
        
        tracker.end_run()

# Find best
best = tracker.get_best_run('accuracy', mode='max')
print(f"✓ Logged {len(tracker.experiments)} experiments")
print(f"\nBest run:")
print(f"  Learning rate: {best['parameters']['learning_rate']}")
print(f"  Batch size: {best['parameters']['batch_size']}")
print(f"  Accuracy: {best['metrics']['accuracy'][-1]['value']:.4f}")

## Advanced Implementation: MLflow Integration

In [ ]:
import numpy as np
from typing import Tuple

# Simulating MLflow API
class MLflowSimulator:
    """Simulates MLflow experiment tracking"""
    
    def __init__(self):
        self.runs = []
        self.current_run = None
    
    def set_experiment(self, name: str):
        """Set active experiment"""
        self.current_experiment = name
    
    def start_run(self):
        """Start a new run"""
        self.current_run = {
            'experiment': self.current_experiment,
            'params': {},
            'metrics': {},
        }
    
    def log_param(self, key: str, value):
        if self.current_run:
            self.current_run['params'][key] = value
    
    def log_metric(self, key: str, value: float):
        if self.current_run:
            self.current_run['metrics'][key] = value
    
    def end_run(self):
        if self.current_run:
            self.runs.append(self.current_run)
            self.current_run = None
    
    def search_runs(self, **filters) -> list:
        """Search runs with filters"""
        results = self.runs
        
        for key, value in filters.items():
            if key.startswith('metric_'):
                metric = key.replace('metric_', '')
                op = '>'
                results = [r for r in results if r['metrics'].get(metric, 0) > value]
            elif key.startswith('param_'):
                param = key.replace('param_', '')
                results = [r for r in results if r['params'].get(param) == value]
        
        return results

# Usage: hyperparameter grid search with MLflow
mlflow = MLflowSimulator()
mlflow.set_experiment('fraud_model_search')

# Grid search: 3 learning rates × 3 regularization values = 9 trials
learning_rates = [0.001, 0.01, 0.1]
reg_strengths = [0.0001, 0.001, 0.01]

print("Running 9 experiments (3×3 grid search):")
for lr in learning_rates:
    for reg in reg_strengths:
        mlflow.start_run()
        mlflow.log_param('learning_rate', lr)
        mlflow.log_param('regularization', reg)
        
        # Simulated metrics
        accuracy = 0.88 + 0.02 * (lr / 0.1) - 0.01 * np.log(reg + 1)
        mlflow.log_metric('accuracy', accuracy)
        mlflow.log_metric('precision', accuracy - 0.01)
        mlflow.log_metric('recall', accuracy - 0.02)
        
        mlflow.end_run()

print(f"✓ Logged {len(mlflow.runs)} runs")

# Search: find best run by accuracy
best_runs = sorted(mlflow.runs, key=lambda r: r['metrics']['accuracy'], reverse=True)
best = best_runs[0]

print(f"\nBest run (by accuracy):")
print(f"  Learning rate: {best['params']['learning_rate']}")
print(f"  Regularization: {best['params']['regularization']}")
print(f"  Accuracy: {best['metrics']['accuracy']:.4f}")

# Advanced: find all runs with accuracy > 0.90
good_runs = [r for r in mlflow.runs if r['metrics']['accuracy'] > 0.90]
print(f"\n✓ Found {len(good_runs)} runs with accuracy > 0.90")

## Real-World Example 1: Netflix Experiment Organization

In [ ]:
import pandas as pd
import numpy as np

def netflix_experiment_tracking():
    """Track and organize 1000+ experiments per month"""

    print("NETFLIX: Experiment Tracking at Scale")
    print("=" * 60)

    # Simulate experiment results
    np.random.seed(42)
    experiments = pd.DataFrame({
        'run_id': [f'exp_{i:04d}' for i in range(50)],
        'model': np.repeat(['ranking', 'matching', 'diversity'], [20, 15, 15]),
        'learning_rate': np.random.loguniform(0.0001, 0.1, 50),
        'batch_size': np.random.choice([32, 64, 128, 256], 50),
        'accuracy': np.random.uniform(0.88, 0.96, 50),
    })

    print(f"\nTotal experiments: {len(experiments)}")
    print(f"Models tested: {experiments['model'].nunique()}")
    print(f"Best accuracy: {experiments['accuracy'].max():.4f}")

    print("\nBRICK ORGANIZATION:")
    for model in experiments['model'].unique():
        runs = experiments[experiments['model'] == model]
        print(f"  {model}: {len(runs)} runs, best accuracy: {runs['accuracy'].max():.4f}")

    print("\nSEARCH & REPRODUCE:")
    best = experiments.loc[experiments['accuracy'].idxmax()]
    print(f"  Best run: {best['run_id']}")
    print(f"  Model: {best['model']}")
    print(f"  Hyperparameters: lr={best['learning_rate']:.4f}, bs={best['batch_size']}")
    print(f"  Accuracy: {best['accuracy']:.4f}")
    print(f"  To reproduce: git checkout <commit>, load hyperparams from tracking DB")

    print("\nSCALE:")
    print("  Current: 50 runs shown (sample)")
    print("  Actual: 1000+ runs/month across 100 models")
    print("  Storage: metadata only = 100KB/run, 100GB/month")
    print("  Search: <1s to find best run by any metric")

netflix_experiment_tracking()


## Real-World Example 2: Stripe Fraud Model Experiments

In [ ]:
import pandas as pd
import numpy as np

def stripe_ab_testing():
    """A/B test fraud model threshold"""

    print("STRIPE: Fraud Model A/B Testing")
    print("=" * 60)

    np.random.seed(42)

    # Control: old model (threshold 0.5)
    control = pd.DataFrame({
        'threshold': [0.5] * 1000,
        'true_positives': np.random.binomial(1, 0.8, 1000),
        'false_positives': np.random.binomial(1, 0.02, 1000),
    })

    # Treatment: new model (threshold 0.3, higher recall)
    treatment = pd.DataFrame({
        'threshold': [0.3] * 1000,
        'true_positives': np.random.binomial(1, 0.95, 1000),
        'false_positives': np.random.binomial(1, 0.05, 1000),
    })

    control_precision = control['true_positives'].sum() / (control['true_positives'].sum() + control['false_positives'].sum())
    control_recall = control['true_positives'].sum() / 1000

    treatment_precision = treatment['true_positives'].sum() / (treatment['true_positives'].sum() + treatment['false_positives'].sum())
    treatment_recall = treatment['true_positives'].sum() / 1000

    print(f"\nCONTROL (Threshold 0.5):")
    print(f"  Precision: {control_precision:.3f}")
    print(f"  Recall: {control_recall:.3f}")
    print(f"  User impact: {control['false_positives'].sum()} false declines")

    print(f"\nTREATMENT (Threshold 0.3):")
    print(f"  Precision: {treatment_precision:.3f}")
    print(f"  Recall: {treatment_recall:.3f}")
    print(f"  User impact: {treatment['false_positives'].sum()} false declines")

    print(f"\nTRADE-OFF ANALYSIS:")
    print(f"  Fraud caught: +{treatment['true_positives'].sum() - control['true_positives'].sum()}")
    print(f"  False declines: +{treatment['false_positives'].sum() - control['false_positives'].sum()}")
    print(f"  Decision: {('Accept' if treatment_recall > 0.90 else 'Reject')} treatment")
    print(f"  Reason: Higher recall worth extra false declines")

stripe_ab_testing()


## Real-World Example 3: Uber Model Experimentation

In [ ]:
def uber_experiment_tracking():
    print("Uber: Distributed Experiment Tracking")
    print()
    
    print("1. Scale:")
    print("   - 3 domains: pricing, matching, ETA")
    print("   - 50+ experiments/day per domain")
    print("   - 100+ experiments/day total")
    print()
    
    print("2. Organization:")
    print("   - Experiments grouped by domain, then by model")
    print("   - Tags: region (US, EU, Asia), model_type, dataset_version")
    print("   - Dashboard: compare experiments within and across domains")
    print()
    
    print("3. Coordination:")
    print("   - Shared infrastructure (experiment tracking system)")
    print("   - Common metrics (latency, accuracy, cost)")
    print("   - Central dashboard for leadership visibility")
    print()
    
    print("4. Results:")
    print("   - Best models deployed via canary (5% → 50% → 100%)")
    print("   - Canary alerts on metric degradation")
    print("   - Auto-rollback if metrics drop >2%")
    print()
    
    print("5. Impact:")
    print("   - 2-3% improvement in core metrics (accuracy, latency)")
    print("   - Faster iteration: from weeks to days")
    print("   - Better collaboration: teams share learnings")

uber_experiment_tracking()

## Interview Case Study: Organizing 100 Experiments/Day

**Scenario:** Your team runs 100+ experiments per day during hyperparameter search. Design an experiment tracking system.

In [ ]:
print("INTERVIEW CASE STUDY: EXPERIMENT ORGANIZATION AT SCALE")
print()

print("CONSTRAINTS:")
print("  - 100+ experiments/day")
print("  - 5+ engineers running experiments simultaneously")
print("  - Need to find 'that run from 2 weeks ago with 95% accuracy'")
print("  - Storage: 1000s of runs × 10GB artifacts = 10TB+ per month")
print()

print("SOLUTION: MLflow-based Tracking System")
print()

print("1. STRUCTURE:")
print("   Experiment: 'recommendation_ranking_v2'")
print("   ├─ Run 1: learning_rate=0.001, batch_size=32 → acc=0.88")
print("   ├─ Run 2: learning_rate=0.01, batch_size=64 → acc=0.94")
print("   ├─ Run 3: learning_rate=0.1, batch_size=128 → acc=0.91")
print("   └─ ... (100 more runs)")
print()

print("2. WHAT TO LOG:")
print("   Parameters: learning_rate, batch_size, dropout, regularization")
print("   Metrics: training_loss, val_accuracy, val_precision")
print("   Artifacts: model_weights (optional, on-demand)")
print("   Tags: model_type, dataset_version, owner")
print("   Metadata: git_commit, training_time, compute_cost")
print()

print("3. SEARCH & COMPARE:")
print("   Query: accuracy >= 0.90")
print("   Result: 15 runs found")
print("   Sort: by accuracy descending")
print("   Best: learning_rate=0.01, batch_size=64, accuracy=0.95")
print()

print("4. REPRODUCIBILITY:")
print("   Run metadata includes:")
print("   - Code version (git commit)")
print("   - Data version (training_set_v5)")
print("   - Dependencies (transformers==4.30)")
print("   - Seed (42)")
print("   Can rerun: git checkout abc123, load training_set_v5, use seed 42")
print()

print("5. STORAGE MANAGEMENT:")
print("   100 runs/day × 30 days = 3000 runs/month")
print("   Retention: keep 1 month hot, 1 year archive")
print("   Artifacts: lazy load (don't download unless needed)")
print("   Cost: metadata only (~100KB/run) is negligible")
print()

print("STRONG ANSWER STRUCTURE:")
print("  'I'd use MLflow with hierarchical organization.")
print("  Log parameters (learning_rate, batch_size), metrics (accuracy),")
print("  tags (dataset_version), and metadata (git_commit, seed).")
print("  Dashboard enables filtering and comparison.")
print("  Find best run: query by metric, sort, inspect parameters.")
print("  Reproducibility: metadata captured enables exact rerun.")
print("  Storage: keep 1 month hot, archive older, lazy load artifacts.")
print("  Scale: supports 100+/day on single server, distributed for larger.'"

## Key Takeaways

**2-Minute Elevator Pitch:**
"Experiment tracking logs parameters, metrics, and metadata for every model training run. Tools like MLflow enable searching, comparing, and reproducing experiments. Essential for hyperparameter optimization: search 1000 configurations, find best 5%, inspect parameters, understand what worked. Without tracking, best models are lost."

**Why It Matters:**
- ✓ Find best experiments quickly ("95% accuracy from 2 weeks ago")
- ✓ Avoid re-running experiments (saved compute)
- ✓ Share learnings across team ("optimal learning_rate is X")
- ✓ Reproduce results (metadata enables exact rerun)

**What to Log:**
- Parameters: all hyperparameters (learning_rate, batch_size, etc.)
- Metrics: validation accuracy, loss, precision, recall
- Tags: dataset_version, model_type, owner
- Metadata: git commit, training_time, seed

**Common Follow-Ups:**
- "100 experiments/day, storage exploding" → Archive old runs, lazy load artifacts
- "Can't find that run from 2 weeks ago" → Better tagging, search functionality
- "Multiple teams running experiments" → Central tracking, permissions, shared dashboards